# EfficientNet-B4 features
Turns every face crop into 1792 numbers with a frozen EfficientNet-B4 (ImageNet weights, `tf_efficientnet_b4.in1k` from timm).

In [1]:
config = {
    "limit": 0,
    "model": "tf_efficientnet_b4.in1k",
    "batch_size": 64,
    "workers": 4,
    "out": "/kaggle/working/features",
}
parts = ["ffpp/train", "ffpp/val", "ffpp/test", "celebdf/test"]

In [2]:
!pip install -q timm==1.0.15

In [3]:
import os
import glob

import numpy as np
import pandas as pd
import torch
from PIL import Image
import timm

In [4]:
def find(pattern):
    for depth in range(8):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + pattern)
        if len(hits) > 0:
            return hits[0]
    raise FileNotFoundError(pattern)


class Crops(torch.utils.data.Dataset):
    def __init__(self, paths, preprocess):
        self.paths = paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.preprocess(Image.open(self.paths[i]).convert("RGB"))

`num_classes=0` removes the classifier head, so the model returns the 1792 pooled features.

In [5]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
model = timm.create_model(config["model"], pretrained=True, num_classes=0)
model = model.to(device).eval()

data_config = timm.data.resolve_model_data_config(model)
data_config["crop_pct"] = 1.0
preprocess = timm.data.create_transform(**data_config, is_training=False)
print(data_config)
print(preprocess)
print("features per crop:", model.num_features)

model.safetensors: reconstructing file:   0%|          |  0.00B / 77.9MB            

model.safetensors: downloading bytes:           |  0.00B            

{'input_size': (3, 380, 380), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 1.0, 'crop_mode': 'center'}
Compose(
    Resize(size=380, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(380, 380))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)
features per crop: 1792


In [6]:
os.makedirs(config["out"], exist_ok=True)
for part in parts:
    name = part.replace("/", "_")
    rows = pd.read_csv(find(name + "_manifest.csv"), dtype={"video_id": str})
    root = find(part)[:-len(part)]
    if config["limit"] > 0:
        videos = rows[["method", "video_id"]].drop_duplicates().head(config["limit"])
        rows = rows.merge(videos)
    paths = [os.path.join(root, p) for p in rows["path"]]
    missing = [p for p in paths if not os.path.exists(p)]
    assert len(missing) == 0, missing[:3]

    loader = torch.utils.data.DataLoader(Crops(paths, preprocess), batch_size=config["batch_size"],
                                         num_workers=config["workers"])
    chunks = []
    with torch.no_grad(), torch.autocast("cuda", enabled=(device == "cuda")):
        for batch in loader:
            chunks.append(model(batch.to(device)).float().cpu().numpy())
    features = np.concatenate(chunks)

    np.save(os.path.join(config["out"], name + "_effb4.npy"), features)
    rows.to_csv(os.path.join(config["out"], name + "_rows.csv"), index=False)
    print(name, features.shape, "mean norm %.2f" % np.linalg.norm(features, axis=1).mean())

ffpp_train (46076, 1792) mean norm 8.49
ffpp_val (22396, 1792) mean norm 8.46
ffpp_test (22400, 1792) mean norm 8.41
celebdf_test (16572, 1792) mean norm 8.42


Quick check that the rows match the CLIP run, so both backbones can be compared crop by crop. Add the CLIP features dataset as an input to run this cell; it is skipped if the CLIP rows are not found.

In [7]:
for part in parts:
    name = part.replace("/", "_")
    mine = pd.read_csv(os.path.join(config["out"], name + "_rows.csv"), dtype={"video_id": str})
    try:
        clip_rows = pd.read_csv(find(name + "_rows.csv"), dtype={"video_id": str})
    except FileNotFoundError:
        print(name, "CLIP rows not attached, skipped")
        continue
    same = len(mine) == len(clip_rows) and (mine["path"].values == clip_rows["path"].values).all()
    print(name, "rows match CLIP:", same)

ffpp_train rows match CLIP: True
ffpp_val rows match CLIP: True
ffpp_test rows match CLIP: True
celebdf_test rows match CLIP: True
